# Τελική εργασία
Ονομα: Μερτζεμεκιανός Μάρκος
ΑΕΜ: 212
Μεταπτυχιακό: Τεχνητή νοημοσύνη

## Επεξεργασία δεδομένων
Σχεδόν σε κάθε στήλη σχεδόν έλειπαν δεδομένα, για την αντιμετώπιση αυτών, χρησιμοποιήθηκε ο SimpleImputer της sklearn. Κάθε στήλη το οποίο είχε κάποιο αλφαριθμητικό σαν δεδομένο χρησιμοποιήθηκε η πιο συχνή τιμή ενώ για δεκαδικούς αριθμούς χρησιμοποιήθηκε ο διάμεσος για να συμπληρωθούν τα δεδομένα.

Επόμενο βήμα που υλοποιήθηκε ήταν η σύμπτηξη των στηλών του αριθμού των παιδιών. Όπου η στήλη "num_children5", "num_children10" και "num_children18". Επίσης αλλάχθηκε η κατηγορία "sworkershh" σε "workers", ενώ δεν κρατήθηκε η στήλη "sfworkershh".

Ακολούθησε έπειτα η κανονικοποίηση των δεδομένων. Οι τιμές που είχαν μόνο δύο τιμές κρατήθηκαν ως έχουν ενώ η κανονικοποίηση στα υπόλοιπα μεγέθη έγινε με το z-score. Συγκεκριμένα χρησιμοποιήθηκε κλάσση StandardScaler.

Τα αντικείμενα που έχουν σαν text χρησιμοποιήθηκε η κωδικοποίηση One Hot. Επειδή οι κατανομές είναι heavily skewed στο τέλος λογαριθμίστηκαν οι έξοδοι.

## Ανάλυση δεδομένων
Οι στήλες θα μπορούσαν να μπουν στις ακόλουθες κατηγορίες
1. Στέγαση: dweltyp, owner, urban
1. Υποδομές: elect, sanitation_source, sewer, water, water_source
1. Εργασία: any_nonagric, edux_max, employed, sector1d, educ_max
1. Τροφή: consumed*

Για την εύρεση κάποιας συσχέτισης με την έξοδο χρησιμοποιήθηκε το correlations barplot.

<figure>
	<img src="correlations_barplot.png" alt="correlations_barplot.png">
	<figcaption align = "center"> Συσχετήσεις με την έξοδο.</figcaption>
</figure>

Οι περισσότερες μεταβλητές δεν είναι καλής ποιότητα καθώς είναι skewed, μόνο η ηλικία δεν ακολουθεί κανονική κατανομή. 

Έγινε και μία προσπάθεια για clustering όπου δεν βγήκε κάποιο συμπέρασμα. Τα labels επεικονίζουν το μέσο όρο και το variance της εξόδου. 

<figure>
	<img src="clustering_results.png" alt="clustering_results.png">
	<figcaption align = "center"> Clusterings </figcaption>
</figure>

## Εφαρμογή αλγορίθμων μηχανικής μάθησης
Οι αλγόριθμου που επιλέχθηκαν είναι οι xgboost, svr και ένα βαθύ νευρονικό δίκτυο.

### xgboost
Ο xgboost επιλέχθηκε διότι του bagging που γίνεται εσωτερικά. Επίσης παρατηρήθηκε μεγαλώνοντας τον αριθμό των δέντρων που χρησιμοποιεί εσωτερικά ο αλγόριθμος έχει καλύτερη απόδοση. Αναλυτικά το grid search έγινε στις παρακάτω παραμέτρους.
- n_estimators: Θέτει τον αριθμό που θα χρησιμοποιηθούν κατά το voting στην εκπαίδευση.
- learning_rate: η μεταβλητή ρυθμίζει την ταχύτητα των βαρών μετά από κάθε επανάληψη κατά το boosting.
- max_depth: είναι το μέγιστο επιτρεπτό βάθος των δέντρων
- subsample: δειγματοληπτεί τα δεδομένα πριν αρχίσει την κατασκευή των δέντρων
- colsample_bytree: δειγματοληπτεί τις στήλες ή χαρακτηριστικά των δειγμάτων
- tree_method (hist): Το hist συγκεκριμένα αλλάζει τον τρόπο που φτιάχνει τα δέντρα. Αντί να εξετάζει κάθε τιμή ξεχωριστά, ομαδοποιεί τα δεδομένα σε 'κουβάδες' (bins). Με αυτόν τον τρόπο, ο αλγόριθμος δεν ψάχνει το καλύτερο σημείο διαχωρισμού ανάμεσα σε εκατομμύρια τιμές, αλλά εστιάζει μόνο στα όρια αυτών των κουβάδων. Αυτό προσφέρει τεράστια ταχύτητα και εξοικονόμηση μνήμης, καθώς υπολογίζει τη σημασία των δεδομένων (gradients) συγκεντρωτικά ανά κουβά.

Χρησιμοποιήθηκε ένα GridSearch με τις ακόλουθες παραμέτρους
- 'n_estimators': [2000, 3000, 4000]
- 'learning_rate': [0.01, 0.05, 0.1]
- 'max_depth': [4, 6, 8, 10]
- 'subsample': [0.7, 0.8, 0.9]
- 'colsample_bytree': [0.7, 0.8, 0.9]
- 'tree_method': ['hist']

Το καλύτερο δέντρο εδώ είχε παραμέτρους 
- MSE: 0.2201 
- 'n_estimators': 4000
- 'learning_rate': 0.01 
- 'max_depth': 6
- 'subsample': 0.7
- 'colsample_bytree': 0.7
- 'tree_method': 'hist'
### SVM
Σαν δεύτερη προσπάθεια χρησιμοποιήθηκε ένας αλγόριθμος support vector machine, πιο συσκεκριμένα την υλοποίηση της cuml τον αλγόριθμο svr. Ο λόγος που επιλέχθηκε είναι γιατί χρησιμοποιεί την κάρτα γραφικών οπότε εκπαιδεύει πάνω στα δείγματα σημαντικά πιο γρήγορα. Σε αυτή την περίπτωση είναι τρεις οι μεταβλητές που έχουν ενδιαφέρον:
- C: To C έχει δύο ρόλους. Ο πρώτος αφορά ότι δεν αφήνει το τελικό επίπεδο να πάρει μεγάλες τιμές, ενώ ο δεύτερος βάζει και κάποια ανοχή ως προς σε ποια μερια του επιπέδου βρίσκονται τα δείγματα της αντίθετης κλάσσης ή σε αυτήν την περίπτωση πόσο μακριά βρίσκονται από τον epsilon-tube.
- epsilon: Το epsilon δηλώνει τι διάμετρο έχει ο σωλήνας.
- gamma: Το γάμμα έχει να κάνει την rbf, και είναι η σταθερά που έχει στον εκθέτη.

Και για αυτήν την μέθοδο χρησιμοποιήθηκε ενα grid search για την εύρεση του καλύτερου regressor.
- 'C': [10, 100]
- 'epsilon': [0.01, 0.1, 0.2]
- 'gamma': ['scale', 0.1]

Το scale διαιρεί με τον αριθμό των χαρακτηρηστικών.

Εδώ βρέθηκε ο svr με παραμέτρους που θα γραφθούν πιο κάτω
{'kernel': 'rbf', 'C': 10, 'epsilon': 0.2, 'gamma': 0.1}


### Deep learning
Σε αυτήν την περίπτωση υλοποιήθηκε ένα mlp με τα ακόλουθα hidden layers.
Οι παράμετροι που χρησιμοποιήθηκαν είναι οι ακόλουθες
- dropout: Το dropout χρησιμοποιείται κατά την εκπαίδευση, σε κάθε κύκλο εκπαίδευσης κλείνει κάποιους νευρώνες ώστε να μην αλληλεξαρτούνται. Αυτό οδηγεί σε καλύτερη γενικοποίηση.
- learning_rate: είναι το relaxation value κατά την εκπαίδευση του linear regression model των νευρώνων στην ουσία ελέγχει την ταχύτητα αλλαγής των βαρών.
- batch_size: είναι το πακετάρισμα των δεδομένων που στέλνεται κάθε φορά για την εκπαίδευση των νευρώνων
- epochs: ελέγχει πόσες φορές γίνεται αυτή η διαδικασία.

Για το grid search χρησιμοποιήθηκαν οι παράμετροι

- [128, 64], 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 32, 'epochs': 100
- [256, 128, 64], 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 100
- [128, 64, 32], 'dropout': 0.1, 'learning_rate': 0.0005, 'batch_size': 32, 'epochs': 100
- [512, 256, 128], 'dropout': 0.3, 'learning_rate': 0.0005, 'batch_size': 64, 'epochs': 100

Τελικά οι καλύτερες παράμετροι βρέθηκαν για
{'hidden_layers': [128, 64, 32], 'dropout': 0.1, 'learning_rate': 0.0005, 'batch_size': 32, 'epochs': 100}



## Επικύρωση αποτελεσμάτων
Για κάθε run χρησιμοποιήθηκε cross validation σαν επικύρωση. Η μετρική που χρησιμοποιήθηκε είναι η RMS. Σε κάθε περίπτωση έγινε ξεχωριστό submission, για κάθε μία από τις τεχνικές που χρησιμοποιήθηκαν παραπάνω. 

<figure>
	<img src="scores.png" alt="scores.png">
	<figcaption align = "center"> Submission</figcaption>
</figure>

Το πρώτο submission έγινε με svr ενώ το δεύτερο με νευρωνικά ενώ το τελευταίο με την xgboost.

Το πρόβλημα είναι σε όλες τις περιπτώσεις είναι ότι η διάσταση των προβλημάτων είναι πολύ
ψηλή. Ένας Kpca αλλά έξυπνα ώστε να κάνουμε συμπίεση του χώρου όσο γίνεται. Και τα τρία μοντέλα δουλεύουν μέτρια. Με μία καλύτερη προεπεξεργασία των δεδομένων ίσως να τα πήγαιναν γενικότερα πιο καλά και τα τρία μοντέλα.

Όσο για το svm θεωρητικά έχει απειρη διάσταση, οπότε περιμέναμε να προσαρμοστεί πιο εύκολα στα δεδομένα πράγμα το οποίο δεν το κατάφερε οπότε το πρόβλημα χρειάζεται αδιαστατοποίηση.
